[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [SQLAlchemy, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)

# Joins and Aggregates &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell builds `scratch/college.db` as the notebook's Setup did, and makes what its worked
examples made: Zoe Nakamura and the empty Fall 2026 section, `POINTS` and `standing`. Run it first.
The tasks do not depend on one another, and the last cell removes the scratch folder.


In [1]:
import logging
import re
import shutil
import warnings
from datetime import date
from pathlib import Path

import sqlalchemy
from sqlalchemy import (CheckConstraint, ForeignKey, MetaData, String, UniqueConstraint, case, create_engine, distinct,
                        event, func, insert, select)
from sqlalchemy.orm import (DeclarativeBase, Mapped, Session, aliased, contains_eager, mapped_column, relationship,
                            selectinload, sessionmaker)
from sqlalchemy.pool import StaticPool

SCRATCH = Path("scratch")
shutil.rmtree(SCRATCH, ignore_errors=True)
SCRATCH.mkdir()
DATABASE = SCRATCH / "college.db"

NAMES = [
    "Ana Reyes", "Ben Okafor", "Chloe Martin", "Daniel Kim", "Elena Petrova", "Felix Wagner",
    "Grace Lin", "Hassan Ali", "Isabel Costa", "Jonas Berg", "Keiko Tanaka", "Liam Murphy",
    "Maya Patel", "Noah Andersen", "Olivia Brandt", "Pavel Novak", "Quinn Harper", "Rosa Delgado",
    "Sam Ito", "Tara Nilsen", "Umar Farouk", "Vera Kowalski", "Wes Carter", "Yara Haddad",
    "Aoife O'Brien",
]
PROGRAMS = ["Biology", "Computer Science", "Mathematics", "Psychology", "History"]
TERMS = [("Fall 2024", "2024-08-26"), ("Spring 2025", "2025-01-13"), ("Fall 2025", "2025-08-25"),
         ("Spring 2026", "2026-01-12")]
STUDENTS = [(name, f"{name[0]}{name.split()[-1]}@college.edu".lower().replace("'", ""),
             PROGRAMS[i % len(PROGRAMS)], TERMS[i % 3][1]) for i, name in enumerate(NAMES)]
COURSES = [
    ("BIO-101", "Introduction to Biology", "Biology", 4),
    ("CHE-110", "General Chemistry", "Chemistry", 4),
    ("MAT-120", "Calculus I", "Mathematics", 4),
    ("MAT-121", "Calculus II", "Mathematics", 4),
    ("CSC-101", "Programming I", "Computer Science", 3),
    ("CSC-201", "Data Structures", "Computer Science", 3),
    ("ENG-105", "Composition", "English", 3),
    ("HIS-110", "World History", "History", 3),
    ("PSY-101", "Introduction to Psychology", "Psychology", 3),
    ("STA-200", "Statistics", "Mathematics", 3),
]
GRADES = ["A", "A-", "B+", "B", "B-", "C+", "C", "C-", "D", "F"]

# One section of every course in every term, so the section of course c in term t has id (t - 1) * 10 + c.
SECTIONS = [(course, term, 30) for term in range(1, len(TERMS) + 1) for course in range(1, len(COURSES) + 1)]

# Three courses a term for every student, from the term they started. Spring 2026 is under way.
ENROLLMENTS = []
for s in range(len(NAMES)):
    for term in range(s % 3 + 1, len(TERMS) + 1):
        for k in range(3):
            section = (term - 1) * len(COURSES) + (s + term + 3 * k) % len(COURSES) + 1
            if term < len(TERMS):
                ENROLLMENTS.append((s + 1, section, "completed", GRADES[(s * 7 + term * 5 + k * 3) % len(GRADES)]))
            else:
                ENROLLMENTS.append((s + 1, section, "enrolled", None))

class PrintStatements(logging.Handler):
    """Print what an engine logs, leaving out the time: every statement, and the values sent with it."""

    def emit(self, record):
        if record.msg == "[%s] %r":                  # after a statement: how long it took, then its values
            values = repr(record.args[1])
            if values != "()":
                print("    values:", values)
        else:
            for line in record.getMessage().splitlines():
                print("   ", line.rstrip())


sql_log = logging.getLogger("sqlalchemy.engine.Engine")
sql_log.handlers = [PrintStatements()]              # this handler alone, however often the cell runs
sql_log.propagate = False                           # and no handler above it prints the same lines again


def college_engine(path=None, echo=False):
    """An engine for the college's database, in a file or in memory, with foreign keys enforced."""
    if path is None:                                # in memory: one connection, and one database, for every thread
        engine = create_engine("sqlite://", poolclass=StaticPool, echo=echo,
                               connect_args={"check_same_thread": False, "autocommit": False})
    else:
        engine = create_engine(f"sqlite:///{path}", echo=echo, connect_args={"autocommit": False})

    @event.listens_for(engine, "connect")
    def enforce_foreign_keys(dbapi_connection, connection_record):
        dbapi_connection.autocommit = True          # the PRAGMA does nothing inside a transaction,
        dbapi_connection.execute("PRAGMA foreign_keys = ON")
        dbapi_connection.autocommit = False         # and with autocommit=False sqlite3 keeps one open

    return engine

NAMING = {
    "pk": "pk_%(table_name)s",
    "uq": "uq_%(table_name)s_%(column_0_N_name)s",
    "ck": "ck_%(table_name)s_%(constraint_name)s",
    "fk": "fk_%(table_name)s_%(column_0_name)s_%(referred_table_name)s",
    "ix": "ix_%(column_0_label)s",
}


GRADE_POINTS = {"A": 4.0, "A-": 3.7, "B+": 3.3, "B": 3.0, "B-": 2.7, "C+": 2.3, "C": 2.0, "C-": 1.7, "D": 1.0, "F": 0.0}


class Base(DeclarativeBase):
    metadata = MetaData(naming_convention=NAMING)


class Student(Base):
    __tablename__ = "students"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(100))
    email: Mapped[str] = mapped_column(String(200), unique=True)
    program: Mapped[str] = mapped_column(String(50))
    started_on: Mapped[date]

    enrollments: Mapped[list["Enrollment"]] = relationship(back_populates="student", order_by="Enrollment.section_id")

    def __repr__(self):
        return f"Student({self.name!r}, {self.program!r})"


class Course(Base):
    __tablename__ = "courses"
    __table_args__ = (CheckConstraint("credits BETWEEN 1 AND 6", name="credits_range"),)

    id: Mapped[int] = mapped_column(primary_key=True)
    code: Mapped[str] = mapped_column(String(10), unique=True)
    title: Mapped[str] = mapped_column(String(100))
    department: Mapped[str] = mapped_column(String(50))
    credits: Mapped[int]

    sections: Mapped[list["Section"]] = relationship(back_populates="course", order_by="Section.term_id")

    def __repr__(self):
        return f"Course({self.code!r}, {self.credits})"


class Term(Base):
    __tablename__ = "terms"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(20), unique=True)
    starts_on: Mapped[date]

    sections: Mapped[list["Section"]] = relationship(back_populates="term", order_by="Section.course_id")

    def __repr__(self):
        return f"Term({self.name!r})"


class Section(Base):
    __tablename__ = "sections"
    __table_args__ = (UniqueConstraint("course_id", "term_id"), CheckConstraint("capacity > 0", name="capacity_positive"))

    id: Mapped[int] = mapped_column(primary_key=True)
    course_id: Mapped[int] = mapped_column(ForeignKey("courses.id"))
    term_id: Mapped[int] = mapped_column(ForeignKey("terms.id"))
    capacity: Mapped[int]

    course: Mapped["Course"] = relationship(back_populates="sections")
    term: Mapped["Term"] = relationship(back_populates="sections")
    enrollments: Mapped[list["Enrollment"]] = relationship(back_populates="section", order_by="Enrollment.student_id")

    def __repr__(self):
        return f"Section({self.id})"


class Enrollment(Base):
    __tablename__ = "enrollments"
    __table_args__ = (CheckConstraint("status IN ('enrolled', 'completed', 'withdrawn')", name="status_known"),)

    student_id: Mapped[int] = mapped_column(ForeignKey("students.id"), primary_key=True)
    section_id: Mapped[int] = mapped_column(ForeignKey("sections.id"), primary_key=True)
    status: Mapped[str] = mapped_column(String(20), server_default="enrolled")
    grade: Mapped[str | None] = mapped_column(String(2))

    student: Mapped["Student"] = relationship(back_populates="enrollments")
    section: Mapped["Section"] = relationship(back_populates="enrollments")

    @property
    def grade_points(self):
        """The points the grade is worth, or None before there is a grade."""
        return None if self.grade is None else GRADE_POINTS[self.grade]

    def __repr__(self):
        return f"Enrollment(student {self.student_id}, section {self.section_id}, {self.grade!r})"


def build_college(engine):
    """Create the college's tables from the classes, load the lists above into them, and count their rows."""
    Base.metadata.create_all(engine)
    rows = {
        Course: [{"code": code, "title": title, "department": department, "credits": credits}
                 for code, title, department, credits in COURSES],
        Student: [{"name": name, "email": email, "program": program, "started_on": date.fromisoformat(started)}
                  for name, email, program, started in STUDENTS],
        Term: [{"name": name, "starts_on": date.fromisoformat(starts)} for name, starts in TERMS],
        Section: [{"course_id": course, "term_id": term, "capacity": capacity} for course, term, capacity in SECTIONS],
        Enrollment: [{"student_id": student, "section_id": section, "status": status, "grade": grade}
                     for student, section, status, grade in ENROLLMENTS],
    }
    with engine.begin() as conn:
        for cls, values in rows.items():
            conn.execute(insert(cls), values)
        return {cls.__tablename__: conn.execute(select(func.count()).select_from(cls)).scalar_one() for cls in rows}


engine = college_engine(DATABASE)
print("sqlalchemy", sqlalchemy.__version__, "|", DATABASE, "|", build_college(engine))

SessionLocal = sessionmaker(engine)


with SessionLocal.begin() as session:
    session.add(Student(name="Zoe Nakamura", email="znakamura@college.edu", program="Computer Science",
                        started_on=date(2026, 8, 24)))              # admitted for the autumn, with no enrollments yet
    statistics = session.scalars(select(Course).where(Course.code == "STA-200")).one()
    fall = Term(name="Fall 2026", starts_on=date(2026, 8, 24))
    session.add(fall)
    fall.sections.append(Section(course=statistics, capacity=30))     # open, and nobody enrolled yet

with SessionLocal() as session:
    print(session.scalar(select(func.count()).select_from(Student)), "students,",
          session.scalar(select(func.count()).select_from(Section)), "sections")


POINTS = case(                                     # grade points in tenths, so that every sum is a whole number
    {"A": 40, "A-": 37, "B+": 33, "B": 30, "B-": 27, "C+": 23, "C": 20, "C-": 17, "D": 10, "F": 0},
    value=Enrollment.grade,
)


def standing(session, term):
    """For every program: how many students took graded courses in a term, and how many of them made the
    dean's list, a term average of 3.0 or more, or fell below 2.0."""
    per_student = (
        select(Enrollment.student_id,
               func.sum(POINTS * Course.credits).label("quality"),
               func.sum(Course.credits).label("credits"))
        .join(Enrollment.section)
        .join(Section.course)
        .join(Section.term)
        .where(Term.name == term, Enrollment.grade.is_not(None))
        .group_by(Enrollment.student_id)
        .subquery()
    )
    deans_list = case((per_student.c.quality >= 30 * per_student.c.credits, 1), else_=0)
    below_two = case((per_student.c.quality < 20 * per_student.c.credits, 1), else_=0)
    report = (
        select(Student.program, func.count(), func.sum(deans_list), func.sum(below_two))
        .join(per_student, per_student.c.student_id == Student.id)
        .group_by(Student.program)
        .order_by(Student.program)
    )
    return session.execute(report).all()


sqlalchemy 2.0.54 | scratch/college.db | {'courses': 10, 'students': 25, 'terms': 4, 'sections': 40, 'enrollments': 228}
26 students, 41 sections


**1.** Sections with fewer than eight students.


In [2]:
SMALL = (
    select(Section.id, Course.code, func.count(Enrollment.student_id).label("students"))
    .join(Section.course)
    .outerjoin(Section.enrollments)
    .where(Section.term_id == 4)
    .group_by(Section.id)
    .having(func.count(Enrollment.student_id) < 8)
    .order_by(Section.id)
)
with SessionLocal() as session:
    print(session.execute(SMALL).all())


[(33, 'MAT-120', 7), (34, 'MAT-121', 7), (36, 'CSC-201', 7), (37, 'ENG-105', 7), (40, 'STA-200', 7)]


`having()` tests the count of every group after the grouping, which `where()` cannot do, since the
count does not exist until the rows are grouped.


**2.** Students with an A, with `any()`.


In [3]:
WITH_AN_A = select(Student.name).where(Student.enrollments.any(Enrollment.grade == "A")).order_by(Student.name)
with SessionLocal() as session:
    names = session.scalars(WITH_AN_A).all()
print(len(names), "students:", names)


12 students: ['Ana Reyes', 'Ben Okafor', 'Felix Wagner', 'Grace Lin', 'Hassan Ali', 'Keiko Tanaka', 'Maya Patel', 'Pavel Novak', 'Quinn Harper', 'Rosa Delgado', 'Vera Kowalski', 'Wes Carter']


**3.** Enrollments in Mathematics courses, with `has()` along a path.


In [4]:
IN_MATHEMATICS = (
    select(func.count())
    .select_from(Enrollment)
    .where(Enrollment.section.has(Section.course.has(Course.department == "Mathematics")))
)
print(" ".join(str(IN_MATHEMATICS.compile(engine)).split()))
with SessionLocal() as session:
    print(session.scalar(IN_MATHEMATICS))


SELECT count(*) AS count_1 FROM enrollments WHERE EXISTS (SELECT 1 FROM sections WHERE sections.id = enrollments.section_id AND (EXISTS (SELECT 1 FROM courses WHERE courses.id = sections.course_id AND courses.department = ?)))
67


`has()` inside `has()` became one `EXISTS` inside another, from the enrollment to its section and
from the section to its course. A join would have answered the same question.


**4.** Every course's grades, averaged in tenths.


In [5]:
PER_COURSE = (
    select(Course.code, func.count(Enrollment.grade).label("grades"), func.sum(POINTS).label("tenths"))
    .join(Course.sections)
    .join(Section.enrollments)
    .where(Enrollment.grade.is_not(None))
    .group_by(Course.code)
    .having(func.count(Enrollment.grade) >= 15)
    .order_by(Course.code)
)
with SessionLocal() as session:
    for code, grades, tenths in session.execute(PER_COURSE):
        print(f"{code}  {grades} grades, averaging {tenths / grades / 10:.2f}")


BIO-101  15 grades, averaging 2.25
CHE-110  15 grades, averaging 2.44
CSC-101  15 grades, averaging 2.25
CSC-201  15 grades, averaging 2.41
ENG-105  16 grades, averaging 2.52
HIS-110  16 grades, averaging 1.90
MAT-120  15 grades, averaging 2.97
MAT-121  15 grades, averaging 1.61
PSY-101  15 grades, averaging 2.82
STA-200  16 grades, averaging 2.31


An average of grades, not of credits, since every grade in one course is worth the same credits. The
sum is whole, and Python divides it once for every course.


**5.** Pairs from one program who started on one day, with `aliased`.


In [6]:
other = aliased(Student)
PAIRS = (
    select(Student.name, other.name, Student.program)
    .join(other, (other.program == Student.program) & (other.started_on == Student.started_on) & (other.id > Student.id))
    .order_by(Student.program, Student.name)
)
with SessionLocal() as session:
    pairs = session.execute(PAIRS).all()
print(len(pairs), "pairs, the first", pairs[:3])


10 pairs, the first [('Ana Reyes', 'Pavel Novak', 'Biology'), ('Felix Wagner', 'Umar Farouk', 'Biology'), ('Ben Okafor', 'Quinn Harper', 'Computer Science')]


`other.id > Student.id` counts every pair once, with the lower id first, and never pairs a student
with themselves.


**6.** Standing for Fall 2024.


In [7]:
with SessionLocal() as session:
    rows = standing(session, "Fall 2024")
for program, students, deans, below in rows:
    noun = "student" if students == 1 else "students"
    print(f"{program:<17} {students} {noun}, {deans} on the dean's list, {below} below 2.0")
print("in all:", sum(row[1] for row in rows))


Biology           2 students, 0 on the dean's list, 0 below 2.0
Computer Science  2 students, 1 on the dean's list, 0 below 2.0
History           2 students, 0 on the dean's list, 1 below 2.0
Mathematics       1 student, 0 on the dean's list, 1 below 2.0
Psychology        2 students, 0 on the dean's list, 1 below 2.0
in all: 9


Nine students, the ones who started in Fall 2024, since nobody else had begun yet.

Last, remove the scratch folder:


In [8]:
engine.dispose()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


---

&#8592; **Back to:** [Joins and Aggregates](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlalchemy-deep-dive/14-joins-and-aggregates.ipynb)  &nbsp;&middot;&nbsp;  [SQLAlchemy, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)
